In [1]:
from datasets import load_dataset
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [2]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]


{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [3]:
print(raw_train_dataset.features)

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


In [4]:
from transformers import AutoTokenizer
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
def tokenize_function(batch):
 return tokenizer(
 batch["text"], truncation=True, padding=True, return_tensors="pt"
 )
tokenize_function(raw_train_dataset[:2])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': tensor([[  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
          1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813,
          2395,  1005,  1055,  1040, 11101,  2989,  1032,  2316,  1997, 11087,
          1011, 22330,  8713,  2015,  1010,  2024,  3773,  2665,  2153,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101, 18431,  2571,  3504,  2646,  3293, 13395,  1006, 26665,  1007,
         26665,  1011,  2797,  5211,  3813, 18431,  2571,  2177,  1010,  1032,
          2029,  2038,  1037,  5891,  2005,  2437,  2092,  1011, 22313,  1998,
          5681,  1032,  6801,  3248,  1999,  1996,  3639,  3068,  1010,  2038,
          5168,  2872,  1032,  2049, 29475,  2006,  2178,  2112,  1997,  1996,
          3006,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 

In [5]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [7]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [8]:
import evaluate
accuracy = evaluate.load("accuracy")
print(accuracy.description)
print(accuracy.compute(references=[0, 1, 0, 1], predictions=[1, 0, 0, 1]))


Accuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:
Accuracy = (TP + TN) / (TP + TN + FP + FN)
 Where:
TP: True positive
TN: True negative
FP: False positive
FN: False negative

{'accuracy': 0.5}


In [10]:
f1_score = evaluate.load("f1")
def compute_metrics(pred):
 labels = pred.label_ids
 preds = pred.predictions.argmax(-1)
 # Compute accuracy and F1 Score
 acc_result = accuracy.compute(references=labels, predictions=preds)
 acc = acc_result["accuracy"]
 f1_result = f1_score.compute(
 references=labels, predictions=preds, average="weighted"
 )
 f1 = f1_result["f1"]
 return {"accuracy": acc, "f1": f1}

In [12]:
!pip install genaibook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [13]:
import torch
from transformers import AutoModelForSequenceClassification
from genaibook.core import get_device
device = get_device()
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
 checkpoint, num_labels=num_labels
).to(device)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
from huggingface_hub import login

login()

In [14]:
from transformers import TrainingArguments
batch_size = 32 # You can change this if you have a big or small GPU
training_args = TrainingArguments(
 "classifier-chapter4",
 push_to_hub=True,
 num_train_epochs=2,
 eval_strategy="epoch",
 per_device_train_batch_size=batch_size,
 per_device_eval_batch_size=batch_size,
)

In [18]:
from transformers import Trainer
# Shuffle the dataset and pick 10,000 examples for training
shuffled_dataset = tokenized_datasets["train"].shuffle(seed=42)
small_split = shuffled_dataset.select(range(10000))
# Initialize the Trainer
trainer = Trainer(
 model=model,
 args=training_args,
 compute_metrics=compute_metrics,
 train_dataset=small_split,
 eval_dataset=tokenized_datasets["test"],
 processing_class=tokenizer,
)

In [ ]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.push_to_hub()


In [ ]:
from transformers import AdamW, get_scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
lr_scheduler = get_scheduler("linear", ...)
for epoch in range(num_epochs):
  for batch in train_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    lr_scheduler.step()
    optimizer.zero

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
pipe = pipeline(
 "text-classification",
 model="genaibook/classifier-chapter4",
 device=device,
)
pipe(
 """The soccer match between Spain and
Portugal ended in a terrible result for Portugal."""
)


In [ ]:
# Get prediction for all samples
model_preds = pipe.predict(tokenized_datasets["test"]["text"])
# Get the dataset labels
references = tokenized_datasets["test"]["label"]
# Get the list of label names
label_names = raw_train_dataset.features["label"].names
# Print results of the first 3 samples
samples = 3
texts = tokenized_datasets["test"]["text"][:samples]
for pred, ref, text in zip(model_preds[:samples], references[:samples], texts):
 print(f"Predicted {pred['label']}; Actual {label_names[ref]};")
 print(text)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
# Convert predicted labels to ids
label_to_id = {name: i for i, name in enumerate(label_names)}
pred_labels = [label_to_id[pred["label"]] for pred in model_preds]
# Compute confusion matrix
confusion_matrix = evaluate.load("confusion_matrix")
cm = confusion_matrix.compute(
 references=references, predictions=pred_labels, normalize="true"
)["confusion_matrix"]
# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
plt.title("Normalized confusion matrix")
plt.show()

In [ ]:
filtered_datasets = raw_datasets.filter(lambda example: example["label"] == 2)
filtered_datasets = filtered_datasets.remove_columns("label")


In [ ]:
from transformers import AutoModelForCausalLM
model_id = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = (
 tokenizer.eos_token
) # Needed as SmolLM does not specify padding token.
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

In [ ]:
def tokenize_function(batch):
 return tokenizer(batch["text"], truncation=True)
tokenized_datasets = filtered_datasets.map(
 tokenize_function,
 batched=True,
 remove_columns=["text"], # We only need the input_ids and attention_mask
)
tokenized_datasets

In [ ]:
from transformers import DataCollatorForLanguageModeling
# mlm corresponds to masked language modeling
# and we set it to False as we are not training a masked language model
# but a causal language model
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
samples = [tokenized_datasets["train"][i] for i in range(3)]
for sample in samples:
 print(f"input_ids shape: {len(sample['input_ids'])}")

In [ ]:
out = data_collator(samples)
for key in out:
 print(f"{key} shape: {out[key].shape}")

In [ ]:
training_args = TrainingArguments(
 "business-news-generator",
 push_to_hub=True,
 per_device_train_batch_size=8,
 weight_decay=0.1,
 lr_scheduler_type="cosine",
 learning_rate=5e-4,
 num_train_epochs=2,
 eval_strategy="steps",
 eval_steps=200,
 logging_steps=200,
)

In [ ]:
trainer = Trainer(
 model=model,
 processing_class=tokenizer,
 args=training_args,
 data_collator=data_collator,
 train_dataset=tokenized_datasets["train"].select(range(5000)),
 eval_dataset=tokenized_datasets["test"],
)
trainer.train()


In [ ]:
trainer.push_to_hub()

In [ ]:
from transformers import pipeline
pipe = pipeline(
 "text-generation",
 model="genaibook/business-news-generator",
 device=device,
)
print(
 pipe("Q1", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
 "generated_text"
 ]
)
print(
 pipe("Wall", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
 "generated_text"
 ]
)
print(
 pipe("Google", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
 "generated_text"
 ]
)


In [ ]:
from peft import LoraConfig, get_peft_model
peft_config = LoraConfig(
 r=8, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM"
)
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

In [ ]:
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
 "gpt2", quantization_config=quantization_config
)

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
 "mistralai/Mistral-7B-v0.3",
quantization_config=quantization_config,
 device_map="auto",
)

In [ ]:
from trl import SFTConfig, SFTTrainer
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")
peft_config = LoraConfig(
 r=8,
 lora_alpha=16,
 lora_dropout=0.05,
 task_type="CAUSAL_LM",
)
sft_config = SFTConfig(
 "fine_tune_e2e",
 push_to_hub=True,
 per_device_train_batch_size=8,
 weight_decay=0.1,
 lr_scheduler_type="cosine",
 learning_rate=5e-4,
 num_train_epochs=2,
 eval_strategy="steps",
 eval_steps=200,
 logging_steps=200,
 gradient_checkpointing=True,
 max_seq_length=512,
 # New parameters
 dataset_text_field="text",
 packing=True,
)
trainer = SFTTrainer(
 model,
 args=sft_config,
 train_dataset=dataset.select(range(300)),
 peft_config=peft_config,
)
trainer.train()
trainer.push_to_hub()

In [ ]:
# We load the base model just as before
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")
model = AutoModelForCausalLM.from_pretrained(
 "mistralai/Mistral-7B-v0.3",
 torch_dtype=torch.float16,
 device_map="auto",
)
# You can load the adapter with `load_adapter`
model.load_adapter("genaibook/fine_tune_e2e") # change with your adapter name
# Alternatively, you could just use `from_pretrained` with the adapter name and
# it will automatically take care of loading the base and adapter models.
# model = AutoModelForCausalLM.from_pretrained("genaibook/fine_tune_e2e"...
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
pipe("### Human: Hello!### Assistant:", max_new_tokens=100)

In [ ]:
pipe = pipeline(
 "text-generation", "HuggingFaceTB/SmolLM-135M-Instruct", device=device
)
messages = [
 {
 "role": "system",
 "content": """You are a friendly chatbot who always responds
 in the style of a pirate""",
 },
 {
 "role": "user",
 "content": "How many helicopters can a human eat in one sitting?",
 },
]
print(pipe(messages, max_new_tokens=128)[0]["generated_text"][-1])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M-Instruct")
chat = [
 {"role": "user", "content": "Hello, how are you?"},
 {
 "role": "assistant",
 "content": "I'm doing great. How can I help you today?",
 },
 {
 "role": "user",
 "content": "I'd like to show off how chat templating works!",
 },
]
tokenizer.apply_chat_template(chat, tokenize=False)

In [ ]:
print(tokenizer.apply_chat_template(chat, tokenize=False))

In [ ]:
def embed_documents(documents: List[str]):
 # Use a sentence transformer model to encode the documents
 # Store the documents somewhere
def retrieve_documents(query: str):
 # Use the stored documents to retrieve
 # the most similar documents to the query
def generate_response(query: str, documents: List[str]):
 # Use the LLM to generate a response
def pipeline(query: str):
 documents = retrieve_documents(query)
 response = generate_response(query, documents)
 return respons